<a href="https://colab.research.google.com/github/dan-zeman/belo-horizonte/blob/main/Udapi3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Universal Dependencies with `udapi-python`

This is the third session with the `udapi-python` library. As we start with a blank virtual machine, we must start again with installing Udapi and downloading the treebank data.

## 1. Install `udapi`

First, we need to install the `udapi` Python library. This can be done using `pip`.

In [1]:
%%bash
# (Preceding a single line with ! makes that line interpreted by shell instead of python.
# Inserting %%bash at the beginning of the cell makes the whole cell interpreted by bash.)
pip install --upgrade udapi
udapy -h

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 8.8 MB/s eta 0:00:00
usage: udapy [optional_arguments] scenario

udapy - Python interface to Udapi - API for Universal Dependencies

Examples of usage:
  udapy -s read.Sentences udpipe.En < in.txt > out.conllu
  udapy -T < sample.conllu | less -R
  udapy -HAM ud.MarkBugs < sample.conllu > bugs.html

positional arguments:
  scenario              A sequence of blocks and their parameters.

options:
  -h, --help            show this help message and exit
  -q, --quiet           Warning, info and debug messages are suppressed. Only fatal errors are reported.
  -v, --verbose         Warning, info and debug messages are printed to the STDERR.
  -s, --save            Add write.Conllu to the end of the scenario
  -T, --save_text_mode_trees
                        Add write.TextModeTrees color=1 to the end of the scenario
  -H, --save_html       Add write.TextModeTreesHtml color=1 to the end of the scenario
  -A, --save_all_attributes
 

## 2. Download a UD Treebank

Download one or more UD treebanks from GitHub. Feel free to add downloads of other treebanks you are interested in. See https://universaldependencies.org/ for the list of available treebanks. Besides the UD homepage, you can also try https://universaldependencies.org/languages.html, where you can filter languages by family and genus.

Smart tip: If you want to work with your own data instead, click on the Files icon in the left pane of Colab, create a treebank folder on the local disk and upload your CoNLL-U file(s) there. Then you will be able to say that this is the treebank you want to load and process.

In [2]:
%%bash
rm -rf UD_*
git clone https://github.com/UniversalDependencies/UD_Portuguese-Porttinari.git

Cloning into 'UD_Portuguese-Porttinari'...


## 3. Load the Treebank in Python

Now we will access the treebank data from Python. The `udapi.core.document.Document` class is used to load the treebank. You can edit the code cell below and specify the treebank you want to work with (it must be one of the treebanks you downloaded above).

**Important:** In some of the cells below, we modify the data (mark nodes) inside Python's memory. If we want to modify and run those cells again, we may need to reload a clean version of the treebank from the disk first. To achieve that, we will have to go back to this section and run this cell again.

In [3]:
import glob
from udapi.core.document import Document

# Read the CoNLL-U files.
###############################################################################
# TODO: Change this variable to explore different datasets. It must be one of
# those you downloaded above, or saved manually through Colab's Files icon.
treebank = 'UD_Portuguese-Porttinari'

# List all .conllu files in the specified treebank folder.
conllu_files = glob.glob(f"{treebank}/*.conllu")
print(f"Found {len(conllu_files)} CoNLL-U files in {treebank}:")
for file in sorted(conllu_files):
    print(file)
print(f"Loading {treebank}...")
# Each CoNLL-U file will be stored as one Document object.
# Note that some treebanks use the '# newdoc' comment to mark document boundaries within each file.
# That is a different notion of document!
filedocs = []
nsent = 0
for file in sorted(conllu_files):
    doc = Document(filename=file)
    filedocs.append(doc)
    nsent += len(doc.bundles)
print(f"Loaded {nsent} sentences from {treebank}.")


Found 3 CoNLL-U files in UD_Portuguese-Porttinari:
UD_Portuguese-Porttinari/pt_porttinari-ud-dev.conllu
UD_Portuguese-Porttinari/pt_porttinari-ud-test.conllu
UD_Portuguese-Porttinari/pt_porttinari-ud-train.conllu
Loading UD_Portuguese-Porttinari...
Loaded 8418 sentences from UD_Portuguese-Porttinari.


## 4. Tree Examples for Papers in LaTeX (Overleaf)

This is a continuation of Session 2, where we learned how to display selected trees directly in Colab, and how to generate a HTML document with highlighted search results, which can then be downloaded from Colab.

If you are writing an article about syntax and need tree diagrams, Udapi can help you generate them from the treebank. It uses the LaTeX publishing system (you may know it as the Overleaf web service). If you are writing your paper in LaTeX, you may download the source code of the examples and insert it in the source code of your paper. If you are writing it in anything else, you may run LaTeX in Colab, it will generate a PDF with the trees and you can then insert them to your document as vector graphics.

### Install a LaTeX compiler

First, we need to ensure a LaTeX compiler is available. Colab environments don't always come with a full TeX distribution, so we'll install a minimal one using `apt-get`.

In [ ]:
%%bash
apt-get update
apt-get install -y texlive-latex-base texlive-fonts-recommended texlive-latex-extra texlive-xetex

### Use Udapi to select examples and save them as LaTeX

Udapi contains a block `write.Tikz`, which generates code understandable by LaTeX. It could generate such code for all trees in the treebank but that is probably much more than you need, so we will also instruct Udapi to filter the treebank and write only some trees. Examples in papers often need to be short so that they fit on the page, so the example code here demonstrates how to select trees that contain exactly 6 nodes. Naturally, you will also want to add criteria that characterize the examples you intend to show in the paper.

In [ ]:
%%bash
# Create examples.tex, the source code for LaTeX.
# The util.Filter block of Udapi is instructed to keep only trees of exactly 6 nodes.
cat UD_Portuguese-Porttinari/pt_porttinari-ud-test.conllu \
  | udapy util.Filter keep_tree='len(tree.descendants) == 6' \
          write.Tikz > examples.tex
# Compile the source code with LaTeX. This will create examples.pdf.
# Then you can download:
# - examples.tex and insert it to the source code of your paper (Overleaf)
# - examples.pdf and insert it to a document as vector graphic
xelatex examples.tex

### Download the files

Finally, you can download the files (PDF and its LaTeX source).

In [15]:
from google.colab import files

files.download('examples.pdf')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

files.download('examples.tex')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 5. Get the Trees in HTML

Simple searches can be achieved by calling the so-called command-line interface of Udapi; it means that we are calling Udapi as a standalone program (which is actually spelled `udapy`, note the final letter).

The command-line program can save the output as a HTML file. We can then download this file and open it in our browser.

In [ ]:
from google.colab import files

# Run udapy as a standalone program.
# The first part of the command sends our CoNLL-U files as input to the pipeline.
# The util.Mark node='...' part will mark each node that satisfies the condition.
# The -HAM options mean that udapy will output the trees as colored HTML,
# highlighting the marked nodes. The output will contain only trees that contain
# at least one marked node.
!cat UD_Portuguese-Porttinari/*.conllu | udapy -HAM util.Mark node='node.udeprel == "expl" and node.parent.upos == "VERB"' > udapi_output.html

# Offer the file for download so that you can open it in your browser.
# We could display it here in Colab but it would not be practical because the
# file could be very long.
print(f"File 'udapi_output.html' is ready for download.")
files.download('udapi_output.html')


2026-07-23 23:34:34,671 [   INFO] run_blocks - No reader specified, using read.Conllu
2026-07-23 23:34:34,671 [   INFO] run_blocks -  ---- ROUND ----
2026-07-23 23:34:34,671 [   INFO] run_blocks - Executing block read.Conllu {}
2026-07-23 23:34:34,672 [   INFO] try_fast_load - Reading -
2026-07-23 23:34:35,556 [   INFO] run_blocks - Executing block util.Mark node=node.udeprel == "expl" and node.parent.upos == "VERB"
2026-07-23 23:34:40,256 [   INFO] run_blocks - Executing block write.TextModeTreesHtml color=1 attributes=form,lemma,upos,xpos,feats,deprel,misc marked_only=1
File 'udapi_output.html' is ready for download.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>